In [17]:
#goal is to process the data from individual company level to hexagon level information
#1. create new variables: productivity, entropy of industry
#2. get hex level information on companies

In [18]:
import pandas as pd
import numpy as np
from scipy.stats import entropy

In [19]:
#read in company level opten data
opten_path = '../data/data_gen/opten_ceginfo_ar_teaor_large.pkl'

# keep only necessary columns
keep_columns = [
    'oo_cegj_sz', 'letszam_besz17', 'letszam_besz18', 'letszam_besz19',
    'letszam_nav_20_januar', 'letszam_nav_20_februar', 'letszam_nav_20_marcius',
    'letszam_nav_20_aprilis', 'letszam_nav_20_majus', 'letszam_nav_20_junius', 'letszam_ksh',
    'tulaj_lancban_kulfoldi', 'h3_10', 'arbev_2018', 'arbev_2019',
    'brutto_hozzaadott_ertek_2018', 'brutto_hozzaadott_ertek_2019',
    'Megnevezés', 'teaor_kod', 'teaor_2', 'teaor_betu'
]

opten = pd.read_pickle(opten_path)[keep_columns]

In [ ]:
# create productivity measure (revenue per employee), company level.
# replacing 0 with NaN in the denominator
opten['productivity'] = opten['arbev_2018'] / opten['letszam_besz18'].replace(0, np.nan)

In [21]:
# Columns to sum
sum_columns = [
    'letszam_besz17', 'letszam_besz18', 'letszam_besz19',
    'letszam_nav_20_januar', 'letszam_nav_20_februar', 'letszam_nav_20_marcius',
    'letszam_nav_20_aprilis', 'letszam_nav_20_majus', 'letszam_nav_20_junius',
    'letszam_ksh', 'arbev_2018', 'arbev_2019',
    'brutto_hozzaadott_ertek_2018', 'brutto_hozzaadott_ertek_2019'
]

# Columns to count distinct values of
count_columns = ['teaor_kod', 'teaor_2', 'teaor_betu', 'oo_cegj_sz']

# Create aggregation dictionary
agg_dict = {col: 'sum' for col in sum_columns}
agg_dict.update({col: 'nunique' for col in count_columns})

# Group by 'h3_10' and apply the aggregation
grouped_opten_h3 = opten.groupby('h3_10').agg(agg_dict).reset_index()

grouped_opten_h3.rename(columns={
    'teaor_kod': 'n_unique_teaor_kod',
    'teaor_2': 'n_unique_teaor_2',
    'teaor_betu': 'n_unique_teaor_betu',
    'oo_cegj_sz': 'n_companies',
}, inplace=True)

# average productivity within location
avg_productivity = (
    opten.groupby('h3_10')['productivity']
    .mean()
    .reset_index(name='productivity')
)

grouped_opten_h3 = grouped_opten_h3.merge(avg_productivity, on='h3_10', how='left')

# Dominant industry per h3

In [23]:
# Find the weighted (by employee nr) most frequent teaor_betu for each location
grouped = opten.groupby(['h3_10', 'teaor_betu'])['letszam_besz18'].sum().reset_index()

# Find the 'teaor_betu' with the highest 'letszam_besz18' sum for each 'h3_10'
max_weighted_teaor_betu = grouped.loc[grouped.groupby('h3_10')['letszam_besz18'].idxmax()]

# Merge the dominant teaor_betu back in as a new, clearly-named column
grouped_opten_h3 = grouped_opten_h3.merge(
    max_weighted_teaor_betu[['h3_10', 'teaor_betu']].rename(columns={'teaor_betu': 'teaor_betu_dominant'}),
    on='h3_10', how='left'
)

In [24]:
# Find the weighted (by employee nr) most frequent teaor_2 for each location
grouped2 = opten.groupby(['h3_10', 'teaor_2'])['letszam_besz18'].sum().reset_index()

# Find the 'teaor_2' with the highest 'letszam_besz18' sum for each 'h3_10'
max_weighted_teaor2 = grouped2.loc[grouped2.groupby('h3_10')['letszam_besz18'].idxmax()]

# Merge the dominant teaor_2 back in as a new, clearly-named column
grouped_opten_h3 = grouped_opten_h3.merge(
    max_weighted_teaor2[['h3_10', 'teaor_2']].rename(columns={'teaor_2': 'teaor_2_dominant'}),
    on='h3_10', how='left'
)

In [25]:
# Replace zeros with a small positive value as preparation for a log transform
cols_to_prep_for_log = [
    'letszam_besz18', 'letszam_besz19', 'letszam_nav_20_junius',
    'letszam_nav_20_januar', 'arbev_2018', 'brutto_hozzaadott_ertek_2018'
]
grouped_opten_h3[cols_to_prep_for_log] = grouped_opten_h3[cols_to_prep_for_log].replace(0, 1)

In [26]:
# calculate entropy of business activity per hex

# Function to calculate Shannon entropy
# for each h3_10 location, which measures the uncertainty of teaor values within that location
def shannon_entropy(probs):
    return entropy(probs, base=2)

In [27]:
def compute_entropy_by_hex(df, category_col, entropy_col_name):
    """For each h3_10, compute the Shannon entropy of the distribution of category_col
    values among the companies located there."""
    counts = df.groupby('h3_10')[category_col].value_counts().unstack(fill_value=0)
    probs = counts.div(counts.sum(axis=1), axis=0)
    entropies = probs.apply(shannon_entropy, axis=1)
    return entropies.reset_index(name=entropy_col_name)

In [28]:
# Entropy at three levels of TEAOR granularity: full class code, 2-digit division, sector letter
entropies_df = compute_entropy_by_hex(opten, 'teaor_kod', 'entropy')
entropies_2dig_df = compute_entropy_by_hex(opten, 'teaor_2', 'entropy_2dig')
entropies_letter_df = compute_entropy_by_hex(opten, 'teaor_betu', 'entropy_letter')

# Merge the three entropy measures together
entropies_all_df = entropies_df.merge(entropies_2dig_df, on='h3_10', how='outer').merge(
    entropies_letter_df, on='h3_10', how='outer'
)

In [29]:
# Merge the entropy measures onto the hex-level dataset
grouped_opten_h3 = grouped_opten_h3.merge(entropies_all_df, on='h3_10', how='left')

In [30]:
grouped_opten_h3 = pd.get_dummies(grouped_opten_h3, columns=['teaor_betu_dominant'], prefix='industry')

In [31]:
grouped_opten_h3.columns

Index(['h3_10', 'letszam_besz17', 'letszam_besz18', 'letszam_besz19',
       'letszam_nav_20_januar', 'letszam_nav_20_februar',
       'letszam_nav_20_marcius', 'letszam_nav_20_aprilis',
       'letszam_nav_20_majus', 'letszam_nav_20_junius', 'letszam_ksh',
       'arbev_2018', 'arbev_2019', 'brutto_hozzaadott_ertek_2018',
       'brutto_hozzaadott_ertek_2019', 'n_unique_teaor_kod',
       'n_unique_teaor_2', 'n_unique_teaor_betu', 'n_companies',
       'productivity', 'teaor_2_dominant', 'entropy', 'entropy_2dig',
       'entropy_letter', 'industry_A', 'industry_B', 'industry_C',
       'industry_D', 'industry_E', 'industry_F', 'industry_G', 'industry_H',
       'industry_I', 'industry_J', 'industry_K', 'industry_L', 'industry_M',
       'industry_N', 'industry_O', 'industry_P', 'industry_Q', 'industry_R',
       'industry_S'],
      dtype='object')

In [32]:
# merge all opten information at the h3 level into one file and save
grouped_opten_h3.to_pickle('../data/data_gen/opten_hex_level.pkl')

In [ ]:
#summary stat for opten variables per hex
numeric_cols = ['letszam_besz18', 'arbev_2018', 'n_companies', 'productivity', 'entropy_letter']

def p25(x): return x.quantile(0.25)
def p75(x): return x.quantile(0.75)
def p90(x): return x.quantile(0.90)

desc = grouped_opten_h3[numeric_cols].agg(
    ['count', 'mean', 'std', 'min', p25, 'median', p75, p90, 'max', 'skew']
)

print(desc)

# Transpose so variables are rows, stats are columns
desc_t = desc.T

# Round for display
desc_t = desc_t.round(2)
# count should be integer
desc_t['count'] = desc_t['count'].astype(int)

print(desc_t)

# Export to LaTeX
latex_table = desc_t.to_latex(
    caption="Summary statistics of firm-level and hexagon-level variables (OPTEN data).",
    label="tab:opten_summary",
    column_format='l' + 'r' * desc_t.shape[1],
    escape=True,
    float_format="%.2f"
)

print(latex_table)

with open("../output/tables/opten_summary_stats.tex", "w") as f:
    f.write(latex_table)

        letszam_besz18    arbev_2018  n_companies  productivity  \
count      8896.000000  8.896000e+03  8896.000000  8.893000e+03   
mean         89.427046  3.955700e+06     3.305868  3.033278e+04   
std         588.222983  2.816291e+07     5.898340  5.886632e+04   
min           4.000000  1.000000e+00     1.000000  0.000000e+00   
p25           7.000000  8.983600e+04     1.000000  9.278444e+03   
median       15.000000  3.086210e+05     2.000000  1.773370e+04   
p75          45.000000  1.182949e+06     3.000000  3.364975e+04   
p90         143.000000  4.807378e+06     7.000000  6.079657e+04   
max       36686.000000  1.009305e+09   143.000000  1.826234e+06   
skew         43.644935  1.973256e+01     8.927426  1.496179e+01   

        entropy_letter  
count      8896.000000  
mean          0.703971  
std           0.855547  
min           0.000000  
p25           0.000000  
median        0.000000  
p75           1.000000  
p90           2.000000  
max           3.521741  
skew        